# ESIOS PV Generation Units — Exploratory Analysis

Explores the ESIOS registered generation units list, filtering for **Solar PV** plants.
This file is the official REE/ESIOS registry of all generation units connected to the Spanish grid.

**Source:** `ESIOS_PV_generations_units_list.csv` — exported from the REE ESIOS platform

**Goal:** Understand the composition of Spain's PV fleet as registered in ESIOS:
- How many units, total capacity
- Size distribution (utility-scale vs small/distributed)
- Top owners (BRP — Balance Responsible Parties)
- Comparison with GEM tracker data (593 plants ≥10 MW)
- Coverage analysis: what fraction of national capacity do the ESIOS units represent?

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

DATA_DIR = os.path.join(os.getcwd(), 'data')
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'spain_total', 'data')

ESIOS_PATH    = os.path.join(DATA_DIR, 'ESIOS_PV_generations_units_list.csv')
METADATA_PATH = os.path.join(DATA_DIR, 'plant_metadata.csv')

print(f'Data directory: {DATA_DIR}')

## 1. Load and filter for Solar PV

In [ ]:
# Load full ESIOS unit list (semicolon-separated, comma as decimal separator)
all_units = pd.read_csv(ESIOS_PATH, sep=';', decimal=',')

print(f'Total generation units in ESIOS: {len(all_units):,}')
print(f'Columns: {all_units.columns.tolist()}')
print(f'\nProduction types:')
print(all_units['Production Type'].value_counts().to_string())

In [ ]:
# Filter for Solar PV
pv = all_units[all_units['Production Type'] == 'Solar PV'].copy().reset_index(drop=True)

# Rename columns for convenience
pv = pv.rename(columns={
    'UF Code': 'uf_code',
    'EIC Code': 'eic_code',
    'Short Description': 'short_name',
    'Large Description': 'long_name',
    'Maximum Power Capacity MW': 'capacity_mw',
    'Production Type': 'production_type',
    'BRP Code': 'brp_code',
    'UP Code': 'up_code',
})

print(f'Solar PV units: {len(pv):,}')
print(f'Total PV capacity: {pv["capacity_mw"].sum():,.1f} MW ({pv["capacity_mw"].sum()/1000:.1f} GW)')
print(f'\nFirst 10 entries:')
pv.head(10)

## 2. Capacity statistics

In [ ]:
print('=== Capacity distribution (MW) ===')
print(pv['capacity_mw'].describe().round(2))
print(f'\nTotal capacity : {pv["capacity_mw"].sum():>10,.1f} MW')
print(f'Median         : {pv["capacity_mw"].median():>10,.1f} MW')
print(f'Mean           : {pv["capacity_mw"].mean():>10,.1f} MW')

# Percentiles
percentiles = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
print('\nCapacity percentiles:')
for p in percentiles:
    val = pv['capacity_mw'].quantile(p)
    print(f'  P{int(p*100):02d}: {val:>8.1f} MW')

In [ ]:
# Capacity brackets
bins   = [0, 1, 5, 10, 50, 100, 200, 500, float('inf')]
labels = ['<1', '1-5', '5-10', '10-50', '50-100', '100-200', '200-500', '500+']

pv['capacity_bracket'] = pd.cut(pv['capacity_mw'], bins=bins, labels=labels, right=False)

bracket_stats = pv.groupby('capacity_bracket', observed=False).agg(
    n_plants    = ('capacity_mw', 'count'),
    total_mw    = ('capacity_mw', 'sum'),
    mean_mw     = ('capacity_mw', 'mean'),
).round(1)

bracket_stats['pct_plants']   = (bracket_stats['n_plants'] / len(pv) * 100).round(1)
bracket_stats['pct_capacity'] = (bracket_stats['total_mw'] / pv['capacity_mw'].sum() * 100).round(1)

print('=== Capacity bracket analysis ===')
print(bracket_stats.to_string())
print(f'\nTotal: {bracket_stats["n_plants"].sum()} plants, {bracket_stats["total_mw"].sum():,.0f} MW')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) Histogram of capacity
axes[0].hist(pv['capacity_mw'], bins=50, edgecolor='black', color='goldenrod')
axes[0].set_xlabel('Capacity (MW)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Capacity distribution (n={len(pv):,})')
axes[0].axvline(pv['capacity_mw'].median(), color='red', ls='--',
                label=f'median = {pv["capacity_mw"].median():.1f} MW')
axes[0].legend()

# (b) Bracket distribution — number of plants vs capacity share
x = range(len(bracket_stats))
w = 0.35
axes[1].bar([i - w/2 for i in x], bracket_stats['pct_plants'],   w, label='% of plants', color='steelblue')
axes[1].bar([i + w/2 for i in x], bracket_stats['pct_capacity'], w, label='% of capacity', color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha='right')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Plants vs capacity by size bracket')
axes[1].legend()

# (c) Cumulative capacity curve
sorted_cap = pv['capacity_mw'].sort_values(ascending=False).values
cum_cap    = np.cumsum(sorted_cap)
cum_pct    = cum_cap / cum_cap[-1] * 100

axes[2].plot(range(1, len(cum_pct)+1), cum_pct, color='darkgreen', lw=1.5)
axes[2].axhline(80, color='red', ls='--', alpha=0.6, label='80% of capacity')
axes[2].axhline(90, color='orange', ls='--', alpha=0.6, label='90% of capacity')

# Find where 80% and 90% are reached
n_80 = np.searchsorted(cum_pct, 80) + 1
n_90 = np.searchsorted(cum_pct, 90) + 1
axes[2].axvline(n_80, color='red', ls=':', alpha=0.4)
axes[2].axvline(n_90, color='orange', ls=':', alpha=0.4)
axes[2].set_xlabel('Number of plants (sorted by size)')
axes[2].set_ylabel('Cumulative capacity (%)')
axes[2].set_title('Cumulative capacity curve')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'Top {n_80} plants cover 80% of total capacity')
print(f'Top {n_90} plants cover 90% of total capacity')

## 3. Top plants by capacity

In [ ]:
top20 = pv.nlargest(20, 'capacity_mw')[['uf_code', 'short_name', 'long_name', 'capacity_mw', 'brp_code']]
top20 = top20.reset_index(drop=True)
top20.index += 1
print('=== Top 20 Solar PV units by capacity ===')
print(top20.to_string())

## 4. BRP analysis (plant ownership)

In [ ]:
brp_stats = pv.groupby('brp_code').agg(
    n_plants   = ('capacity_mw', 'count'),
    total_mw   = ('capacity_mw', 'sum'),
    avg_mw     = ('capacity_mw', 'mean'),
    max_mw     = ('capacity_mw', 'max'),
).round(1).sort_values('total_mw', ascending=False)

brp_stats['pct_capacity'] = (brp_stats['total_mw'] / pv['capacity_mw'].sum() * 100).round(1)
brp_stats['cum_pct'] = brp_stats['pct_capacity'].cumsum().round(1)

print(f'Total BRPs (owners): {len(brp_stats)}')
print(f'\n=== Top 20 BRPs by total capacity ===')
print(brp_stats.head(20).to_string())

print(f'\nTop 5 BRPs control {brp_stats.head(5)["pct_capacity"].sum():.1f}% of total PV capacity')
print(f'Top 10 BRPs control {brp_stats.head(10)["pct_capacity"].sum():.1f}% of total PV capacity')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Top 15 BRPs by capacity
top_brp = brp_stats.head(15)
axes[0].barh(range(len(top_brp)), top_brp['total_mw'], color='steelblue', edgecolor='black')
axes[0].set_yticks(range(len(top_brp)))
axes[0].set_yticklabels(top_brp.index)
axes[0].set_xlabel('Total capacity (MW)')
axes[0].set_title('Top 15 BRPs by PV capacity')
axes[0].invert_yaxis()

# (b) BRP concentration — cumulative
axes[1].plot(range(1, len(brp_stats)+1), brp_stats['cum_pct'].values, color='darkgreen', lw=1.5)
axes[1].axhline(80, color='red', ls='--', alpha=0.6)
axes[1].set_xlabel('Number of BRPs')
axes[1].set_ylabel('Cumulative capacity (%)')
axes[1].set_title('Market concentration — BRP cumulative capacity')
n_brp_80 = np.searchsorted(brp_stats['cum_pct'].values, 80) + 1
axes[1].axvline(n_brp_80, color='red', ls=':', alpha=0.4, label=f'{n_brp_80} BRPs = 80%')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. UP Code analysis (physical unit groupings)

In [ ]:
up_stats = pv.groupby('up_code').agg(
    n_units    = ('capacity_mw', 'count'),
    total_mw   = ('capacity_mw', 'sum'),
).sort_values('total_mw', ascending=False)

print(f'Unique UP codes: {len(up_stats)}')
print(f'UP codes with multiple units: {(up_stats["n_units"] > 1).sum()}')
print(f'\n=== Top 15 UP codes by capacity ===')
print(up_stats.head(15).to_string())

# Distribution of units per UP code
print(f'\nUnits per UP code distribution:')
print(up_stats['n_units'].describe().round(1))

## 6. Comparison with GEM plant metadata

The GEM Global Solar Power Tracker provides 593 plants ≥10 MW for Spain.
ESIOS has 1,883 Solar PV generation units. Let's compare the two sources.

In [ ]:
# Load GEM-based plant metadata if available
gem_available = os.path.exists(METADATA_PATH)
if gem_available:
    gem = pd.read_csv(METADATA_PATH)
    print(f'GEM plants loaded: {len(gem)}')
    print(f'GEM total capacity: {gem["capacity_mw"].sum():,.0f} MW')
else:
    print('plant_metadata.csv not found — skipping GEM comparison')

if gem_available:
    # ESIOS units >= 10 MW for comparable scope
    pv_large = pv[pv['capacity_mw'] >= 10]

    print(f'\n{"":30s} {"ESIOS (all)":>12s} {"ESIOS (>=10MW)":>15s} {"GEM (>=10MW)":>13s}')
    print(f'{"Number of entries":30s} {len(pv):>12,} {len(pv_large):>15,} {len(gem):>13,}')
    print(f'{"Total capacity (MW)":30s} {pv["capacity_mw"].sum():>12,.0f} {pv_large["capacity_mw"].sum():>15,.0f} {gem["capacity_mw"].sum():>13,.0f}')
    print(f'{"Mean capacity (MW)":30s} {pv["capacity_mw"].mean():>12.1f} {pv_large["capacity_mw"].mean():>15.1f} {gem["capacity_mw"].mean():>13.1f}')
    print(f'{"Max capacity (MW)":30s} {pv["capacity_mw"].max():>12.1f} {pv_large["capacity_mw"].max():>15.1f} {gem["capacity_mw"].max():>13.1f}')

    print(f'\nNote: ESIOS lists individual generation *units* (UF codes), while GEM lists')
    print(f'*projects* which may consist of multiple units/phases. This explains why')
    print(f'ESIOS may have more entries but different total capacity.')

In [ ]:
if gem_available:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # (a) Capacity histograms side-by-side (>=10 MW scope)
    cap_bins = np.arange(0, 650, 25)
    axes[0].hist(pv_large['capacity_mw'], bins=cap_bins, alpha=0.6, label=f'ESIOS (n={len(pv_large)})',
                 color='coral', edgecolor='black')
    axes[0].hist(gem['capacity_mw'], bins=cap_bins, alpha=0.6, label=f'GEM (n={len(gem)})',
                 color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Capacity (MW)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Capacity distribution: ESIOS vs GEM (>=10 MW)')
    axes[0].legend()

    # (b) Capacity coverage: what do small plants add?
    brackets = ['<1 MW', '1-5 MW', '5-10 MW', '10-50 MW', '50-100 MW', '100+ MW']
    bracket_bins = [0, 1, 5, 10, 50, 100, float('inf')]
    pv['size_group'] = pd.cut(pv['capacity_mw'], bins=bracket_bins, labels=brackets, right=False)
    group_cap = pv.groupby('size_group', observed=False)['capacity_mw'].sum()

    axes[1].bar(range(len(brackets)), group_cap.values / 1000, color='goldenrod', edgecolor='black')
    axes[1].set_xticks(range(len(brackets)))
    axes[1].set_xticklabels(brackets, rotation=45, ha='right')
    axes[1].set_ylabel('Total capacity (GW)')
    axes[1].set_title('Capacity by size group (all ESIOS PV units)')

    plt.tight_layout()
    plt.show()

## 7. Summary

In [ ]:
pv_ge10 = pv[pv['capacity_mw'] >= 10]
pv_lt10 = pv[pv['capacity_mw'] < 10]

print('=' * 60)
print('ESIOS Solar PV Generation Units — Summary')
print('=' * 60)
print(f'Total Solar PV units           : {len(pv):>6,}')
print(f'Total capacity                 : {pv["capacity_mw"].sum():>10,.1f} MW ({pv["capacity_mw"].sum()/1000:.1f} GW)')
print(f'')
print(f'Units >= 10 MW (utility-scale)  : {len(pv_ge10):>6,}  ({pv_ge10["capacity_mw"].sum():,.0f} MW = {pv_ge10["capacity_mw"].sum()/pv["capacity_mw"].sum()*100:.1f}% of capacity)')
print(f'Units <  10 MW (small-scale)    : {len(pv_lt10):>6,}  ({pv_lt10["capacity_mw"].sum():,.0f} MW = {pv_lt10["capacity_mw"].sum()/pv["capacity_mw"].sum()*100:.1f}% of capacity)')
print(f'')
print(f'Unique BRPs (owners)           : {pv["brp_code"].nunique():>6,}')
print(f'Unique UP codes (phys. units)  : {pv["up_code"].nunique():>6,}')
print(f'')
print(f'Largest unit                   : {pv["capacity_mw"].max():.1f} MW ({pv.loc[pv["capacity_mw"].idxmax(), "short_name"]})')
print(f'Smallest unit                  : {pv["capacity_mw"].min():.1f} MW')
print(f'Median capacity                : {pv["capacity_mw"].median():.1f} MW')

if gem_available:
    print(f'')
    print(f'--- Comparison with GEM ---')
    print(f'GEM plants (>=10 MW)           : {len(gem):>6,}  ({gem["capacity_mw"].sum():,.0f} MW)')
    print(f'ESIOS units >= 10 MW           : {len(pv_ge10):>6,}  ({pv_ge10["capacity_mw"].sum():,.0f} MW)')
    print(f'Capacity difference            :          {pv_ge10["capacity_mw"].sum() - gem["capacity_mw"].sum():+,.0f} MW')
    print(f'\nNote: Differences arise because ESIOS registers individual generation')
    print(f'units (UFs), while GEM tracks projects (which may bundle multiple UFs).')
    print(f'GEM also deduplicates multi-phase projects into single entries.')